# Tutorial 5: CNN Architecture Experiments
Experiment with different CNN hyperparameters to observe their effect on performance and overfitting.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

# Data Augmentation to help prevent overfitting
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

In [ ]:
class ExperimentalCNN(nn.Module):
    def __init__(self, filter_size=3, num_filters=32, fc_neurons=128):
        super(ExperimentalCNN, self).__init__()
        # Conv Block 1
        self.conv_layer1 = nn.Sequential(
            nn.Conv2d(3, num_filters, kernel_size=filter_size, padding=filter_size//2),
            nn.BatchNorm2d(num_filters),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        # Conv Block 2
        self.conv_layer2 = nn.Sequential(
            nn.Conv2d(num_filters, num_filters * 2, kernel_size=filter_size, padding=filter_size//2),
            nn.BatchNorm2d(num_filters * 2),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        # Fully Connected Block
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(num_filters * 2 * 8 * 8, fc_neurons),
            nn.ReLU(),
            nn.Dropout(0.5), # Prevents overfitting
            nn.Linear(fc_neurons, 10)
        )

    def forward(self, x):
        x = self.conv_layer1(x)
        x = self.conv_layer2(x)
        x = self.classifier(x)
        return x

# Instantiate with custom architecture parameters
model = ExperimentalCNN(filter_size=3, num_filters=32, fc_neurons=128)
print(model)

ExperimentalCNN(
  (conv_layer1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv_layer2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4096, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [ ]:
# Training and Evaluation Setup
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    for inputs, labels in loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    return running_loss / len(loader)

# Run for a few epochs to see performance
for epoch in range(3):
    loss = train_epoch(model, trainloader, optimizer, criterion)
    print(f"Epoch {epoch+1} Loss: {loss:.4f}")

Epoch 1 Loss: 1.6867
Epoch 2 Loss: 1.4820
Epoch 3 Loss: 1.4036


In [ ]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in loader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

original_acc = evaluate(model, testloader)
print(f"Original Model Test Accuracy: {original_acc:.2f}%")

Original Model Test Accuracy: 61.17%


In [ ]:
# Task 1 & 2: Modified Architecture with different filter sizes and more layers
class ModifiedCNN(nn.Module):
    def __init__(self):
        super(ModifiedCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=5, padding=2), # Larger filter size (5x5)
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), # Increased number of filters
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1), # Added extra convolution layer
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)) # Global Average Pooling for better generalization
        )
        self.classifier = nn.Sequential(
            nn.Linear(256, 256), # Modified number of neurons in FC layer
            nn.ReLU(),
            nn.Dropout(0.4), # Task 3: Increased dropout to combat overfitting
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

new_model = ModifiedCNN()
new_optimizer = optim.Adam(new_model.parameters(), lr=0.001)

print("Training Modified Model...")
for epoch in range(3):
    loss = train_epoch(new_model, trainloader, new_optimizer, criterion)
    print(f"Epoch {epoch+1} Loss: {loss:.4f}")

modified_acc = evaluate(new_model, testloader)
print(f"\nModified Model Test Accuracy: {modified_acc:.2f}%")
print(f"Performance Difference: {modified_acc - original_acc:+.2f}%")

Training Modified Model...
Epoch 1 Loss: 1.6460
Epoch 2 Loss: 1.3307
Epoch 3 Loss: 1.1759

Modified Model Test Accuracy: 62.49%
Performance Difference: +1.32%


In [ ]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in loader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

original_acc = evaluate(model, testloader)
print(f"Original Model Test Accuracy: {original_acc:.2f}%")

Original Model Test Accuracy: 60.73%


In [ ]:
# Task 1 & 2: Modified Architecture with different filter sizes and more layers
class ModifiedCNN(nn.Module):
    def __init__(self):
        super(ModifiedCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=5, padding=2), # Task 1: Larger filter size
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), # Task 1: More filters
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1), # Task 1: Extra layer
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Linear(256, 256), # Task 1: Change number of neurons
            nn.ReLU(),
            nn.Dropout(0.3), # Task 3: Regularization for overfitting
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

new_model = ModifiedCNN()
new_optimizer = optim.Adam(new_model.parameters(), lr=0.001)

print("Training Modified Model...")
for epoch in range(3):
    train_epoch(new_model, trainloader, new_optimizer, criterion)

modified_acc = evaluate(new_model, testloader)
print(f"Modified Model Test Accuracy: {modified_acc:.2f}%")
print(f"Accuracy Improvement: {modified_acc - original_acc:.2f}%")

Training Modified Model...
Modified Model Test Accuracy: 60.80%
Accuracy Improvement: 0.07%


In [ ]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in loader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

original_acc = evaluate(model, testloader)
print(f"Original Model Test Accuracy: {original_acc:.2f}%")

Original Model Test Accuracy: 61.23%


In [ ]:
# Task 1: Modified Architecture (Different filters and more layers)
class ModifiedCNN(nn.Module):
    def __init__(self):
        super(ModifiedCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=5, padding=2), # Larger filter size
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1), # Additional Layer
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Linear(256, 256), # More neurons
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

new_model = ModifiedCNN()
new_optimizer = optim.Adam(new_model.parameters(), lr=0.001)

print("Training Modified Model...")
for epoch in range(3):
    train_epoch(new_model, trainloader, new_optimizer, criterion)

modified_acc = evaluate(new_model, testloader)
print(f"Modified Model Test Accuracy: {modified_acc:.2f}%")

Training Modified Model...
Modified Model Test Accuracy: 61.70%
